Step 1: Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
import torch
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

# Set a fixed random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

def create_8x8_matrix(data_row):
    numeric_row = pd.to_numeric(data_row, errors='coerce').fillna(0)
    num_elements_required = 64
    if len(numeric_row) < num_elements_required:
        numeric_row = np.pad(numeric_row, (0, num_elements_required - len(numeric_row)), 'constant')
    matrix = np.array(numeric_row).reshape(8, 8)
    return matrix

def process_dataset(dataset):
    matrices = []
    for _, row in dataset.iterrows():
        matrix = create_8x8_matrix(row)
        matrices.append(matrix)
    return matrices

# Load dataset
df = pd.read_csv('kddcup.data_10_percent.csv', header=None)

# Identify and remove non-numeric columns
non_numeric_columns = [1, 2, 3]
df_features = df.drop(non_numeric_columns, axis=1)

# Process each row to create 8x8 matrices
matrices = process_dataset(df_features)

# Convert to a 3D numpy array
X = np.array(matrices)

# Encode the labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df.iloc[:, -1])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)


Step 2: Building the Model

In [2]:
import torch
import torch.nn as nn

class DeepAutoencoder(nn.Module):
    def __init__(self, num_input_features, num_classes, dropout_rate=0.4):
        super(DeepAutoencoder, self).__init__()
        self.num_input_features = num_input_features

        # Adjust input size for Conv1d with dilation=2
        conv1d_output_size_dilated2 = (num_input_features - 2 * (3 - 1) - 1) + 1  # Adjusted for dilation=2

        # Dilated Encoder with dilation factor of 2
        self.encoder_dilated2 = nn.Sequential(
            nn.Linear(num_input_features, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Conv1d(in_channels=1, out_channels=1, kernel_size=3, stride=1, dilation=2),  # Adjusted dilation
            nn.Flatten(),
            nn.Linear(conv1d_output_size_dilated2, 32)
        )

        # Dilated Decoder with adjusted input and output sizes for dilation=2
        self.decoder_dilated2 = nn.Sequential(
            nn.Linear(32, conv1d_output_size_dilated2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Unflatten(dim=1, unflattened_size=(1, conv1d_output_size_dilated2)),  # Ensure correct input shape for ConvTranspose1d
            nn.ConvTranspose1d(1, 1, kernel_size=3, stride=1, dilation=2),  # Adjusted dilation
            nn.Flatten(),
            nn.Linear(64, num_input_features)
        )

        # Classifier
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, x):
        x = x.view(-1, self.num_input_features)
        encoded_dilated2 = self.encoder_dilated2(x.unsqueeze(1))
        decoded_dilated2 = self.decoder_dilated2(encoded_dilated2)
        classification = self.classifier(encoded_dilated2)
        return decoded_dilated2, classification



# Example usage
num_classes = len(label_encoder.classes_)  # Update this based on your dataset
print(num_classes)
num_input_features = 64  # Update this based on your dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepAutoencoder(num_input_features, num_classes).to(device)


23


Step 3: Training the Model

In [3]:
import torch.optim as optim  # Import the optim module

LEARNING_RATE = 0.001  # Adjusted learning rate
BATCH_SIZE = 64        # Adjusted batch size

criterion_reconstruction = nn.L1Loss()
criterion_classification = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_data = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

num_epochs = 100



for epoch in range(num_epochs):
    model.train()
    running_loss_reconstruction = 0.0
    running_loss_classification = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        decoded, classification = model(inputs.view(-1, num_input_features))
        loss_reconstruction = criterion_reconstruction(decoded, inputs.view(-1, num_input_features))
        loss_classification = criterion_classification(classification, labels)
        loss = loss_reconstruction + loss_classification
        loss.backward()
        optimizer.step()

        running_loss_reconstruction += loss_reconstruction.item()
        running_loss_classification += loss_classification.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Reconstruction Loss: {running_loss_reconstruction / len(train_loader)}, Classification Loss: {running_loss_classification / len(train_loader)}')

Epoch [1/100], Reconstruction Loss: 66.1200612125825, Classification Loss: 29.20291153753247
Epoch [2/100], Reconstruction Loss: 52.314831187605655, Classification Loss: 66.52774717363776
Epoch [3/100], Reconstruction Loss: 37.36404089293538, Classification Loss: 35.58475766924748
Epoch [4/100], Reconstruction Loss: 35.643027185364296, Classification Loss: 17.910670080257074
Epoch [5/100], Reconstruction Loss: 33.67297781917921, Classification Loss: 8.698097352912072
Epoch [6/100], Reconstruction Loss: 46.90622094426954, Classification Loss: 13.982287208275268
Epoch [7/100], Reconstruction Loss: 32.64359995062899, Classification Loss: 39.482311710860834
Epoch [8/100], Reconstruction Loss: 32.84541193500084, Classification Loss: 6.045190497081757
Epoch [9/100], Reconstruction Loss: 31.928243487157147, Classification Loss: 18.071532148656487
Epoch [10/100], Reconstruction Loss: 36.389365262812284, Classification Loss: 17.55509663809137
Epoch [11/100], Reconstruction Loss: 42.889628031257

Step 4: Evaluating the Model and Calculating Metrics

In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import csv

model.eval()
true_labels = []
predicted_labels = []

test_data = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.view(-1, num_input_features)
        inputs, labels = inputs.to(device), labels.to(device)
        decoded, classification = model(inputs)
        _, predicted = torch.max(classification, 1)
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predicted.cpu().tolist())

accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels, average='weighted', zero_division=1)
recall = recall_score(true_labels, predicted_labels, average='weighted', zero_division=1)
f1 = f1_score(true_labels, predicted_labels, average='weighted', zero_division=1)
conf_matrix = confusion_matrix(true_labels, predicted_labels)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:\n", conf_matrix)



csv_file = "kdd99d2o.csv"

# نوشتن داده به فایل CSV
with open(csv_file, mode='w', newline='', encoding='utf-8-sig') as file:
    writer = csv.writer(file)
    writer.writerows(conf_matrix)

print(f"فایل {csv_file} با موفقیت ایجاد شد.")

Accuracy: 0.9852719705925218
Precision: 0.9821668072730907
Recall: 0.9852719705925218
F1 Score: 0.9813826268979314
Confusion Matrix:
 [[  541     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0     0    11
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0     0     1
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0     0    10
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0     0     1
      0     0     2     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0     0   324
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     3     0     2
      